In [ ]:
import requests
from pprint import pprint
import pandas as pd

# connect to the BibMedEd API running in Docker
api_url = "http://localhost:8000/api"

# Get all the projects from the server
projects = requests.get(f"{api_url}/projects").json()
print(projects)

In [ ]:
# As we found 'id":1 from json
project_id = 1

# we need all the API for all publications in this project
publications_res = requests.get(f"http://localhost:8000/api/projects/{project_id}/analysis/publications")
publications_data = publications_res.json()

# let's check it out what we got
pprint(publications_data)

In [ ]:
# retriving year and its number of publishments 
year_count = pd.DataFrame(publications_data["results"]["yearly_counts"]).rename(columns = {"count":"NumberOfPublishedPaper"})

print(year_count)

In [ ]:
# importing matplotlib to plot a line chart
import matplotlib.pyplot as plt 

In [ ]:
# plotting Bar chart
x = year_count["year"]
y = year_count["NumberOfPublishedPaper"]

plt.figure(figsize=(10,6))

bars = plt.bar(x,y, width = 0.65, color = "#4C78A8", edgecolor = "black", linewidth = 0.8)

plt.xlabel('Year', fontsize = 12)
plt.ylabel('Number of Published Papers', fontsize = 12)

plt.title("Published Papers By Years", fontsize = 16, fontweight = "bold")

plt.yticks(range(0, int(y.max()) + 1)) # logic : gives range from 0 upto max and increased by 1 because endpoint of range is exclusive
plt.ylim(0, y.max() * 1.1) # y.max()*1.1 because it adds 10% extra space above the tallest bar.
plt.ylim(0, 2.2)

plt.grid(axis = "y", linestyle = "--", alpha = 0.3)

for bar in bars:
    height = bar.get_height()
    
    plt.text(
        bar.get_x()+ bar.get_width()/2,
        height+0.05,
        str(int(height)),
        ha = "center",
        fontsize = 11,
        fontweight = "bold")
    

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.tight_layout()
plt.show()

In [ ]:
# getting  authors 
authors_res = requests.post(f"{api_url}/projects/{project_id}/analysis/authors")
print(authors_res)

authors_data = authors_res.json()
pprint(authors_data)

In [ ]:
authors_info = pd.DataFrame(authors_data.get("results").get("coauthorship_network").get("nodes")).set_index("id").rename(columns = {"name":"Author's Name","pub_count":"TotalCountOfPublishment"}).sort_values("TotalCountOfPublishment", ascending=False)

pprint(authors_info)

In [ ]:
x = authors_info["Author's Name"]
y = authors_info["TotalCountOfPublishment"]

plt.figure(figsize = (10,6))

bars = plt.barh(x,y,  color = "skyblue", edgecolor = "black", height=  0.8)

plt.xlabel("Total Count Of Publishment", fontsize = 12)
plt.ylabel("Authors", fontsize = 12 )

plt.grid(axis = "x", linestyle = "--", alpha = 0.3)

for bar in bars:
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
             str(int(bar.get_width())), 
             ha="left", va="center", fontsize=11, fontweight="bold")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.title("Top Authors by Publication Count", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
keyword_res = requests.post(f"{api_url}/projects/{project_id}/analysis/keywords")
print(keyword_res)
keyword_data = keyword_res.json()
pprint(keyword_data)

In [ ]:
top_keywords = pd.DataFrame(keyword_data["results"]["top_keywords"]).rename(columns = {"term":"Keywords", "count" : "NumberOfTimesWordAppeared"}).sort_values("NumberOfTimesWordAppeared", ascending = False)
top_keywords

In [ ]:
y = top_keywords["NumberOfTimesWordAppeared"]
x =  top_keywords["Keywords"]

plt.figure(figsize = (10,6))

bars = plt.barh(x,y,  color = "skyblue", edgecolor = "black", height=  0.8)

plt.xlabel("Total Count Of Keyword Appearance", fontsize = 12)
plt.ylabel("Terms", fontsize = 12 )

plt.grid(axis = "x", linestyle = "--", alpha = 0.3)

for bar in bars:
    plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
             str(int(bar.get_width())), 
             ha="left", va="center", fontsize=11, fontweight="bold")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.figtext(
    0.5, 0.01, "Synthetic sample data",ha = "center", fontsize = 10, style = 'italic'
)

plt.title("Top Keywords by Appearance", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
bundle_url = f"http://localhost:8000/api/projects/{project_id}/export/bundle"

bundle_response = requests.get(bundle_url)

with open("sample_project_export.zip", "wb") as f:
    f.write(bundle_response.content)

print(f"Downloaded zip, size: {len(bundle_response.content)} bytes")